In [1]:
import lsdb
from dask.distributed import Client
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import multiband_fit_template_numba as mbft
import nested_pandas as npd
from nested_pandas.utils import count_nested
from scipy.signal import find_peaks

import logging

/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup and DP2 Loading

In [2]:
client=Client(n_workers=6, memory_limit="24GB", threads_per_worker=1, silence_logs=logging.ERROR)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 6
Total threads: 6,Total memory: 134.11 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33385,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36043,Total threads: 1
Dashboard: http://127.0.0.1:37043/status,Memory: 22.35 GiB
Nanny: tcp://127.0.0.1:43465,


2026-06-03 16:42:14,760 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='localhost:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
  File "/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger val

In [3]:
cone = lsdb.ConeSearch(ra=300.0, dec=-25.0, radius_arcsec=70000.0)
usdf_path = "/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/object_collection"
shire_path = "/astro/store/shire/hats/dash/hats/v30_0_6/object_collection"
dp2 = lsdb.open_catalog(shire_path,
                        columns=['objectId', 'coord_dec', 'coord_ra', 'objectForcedSource', 'u_psfMag', 'u_psfMagErr',
                                 'g_psfMag', 'g_psfMagErr', 'r_psfMag', 'r_psfMagErr', 'i_psfMag', 'i_psfMagErr', 'z_psfMag', 'z_psfMagErr',
                                 'y_psfMag', 'y_psfMagErr', 'ebv'],
                        search_filter=cone)

dp2

,objectId,coord_dec,coord_ra,objectForcedSource,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv
npartitions=29184,,,,,,,,,,,,,,,,,
"Order: 6, Pixel: 28753",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow]
"Order: 6, Pixel: 28755",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 196331",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 196332",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


# 1. Filter to RR Lyrae Candidate Subset

In [4]:
# load in the fitter template
tem = mbft.load_template_dir("lsst_template")

# Define Flag Columns
nested_cols = dp2.meta.get_subcolumns("objectForcedSource")
FLAG_COLS = [col for col in nested_cols if 'Flag' in col or 'flag' in col]

In [5]:
# a bunch of helper functions to help us whittle down the big data set
# this one does what it says on the tin
def count_points(df, new_name='n_lc'):
    # Asked to count `lc`, this will add a column called `n_lc`
    return count_nested(df, "objectForcedSource").rename(columns={'n_objectForcedSource':new_name})

def calc_std_partition(df, band='g'):
    df = npd.NestedFrame(df)
    
    # filter to requested band (flags already removed by filter_flags_partition)
    lc = df.query(f"objectForcedSource.band == '{band}'")['objectForcedSource']
    
    if len(lc) == 0:
        return df.assign(std=np.nan)
    
    # vectorized min/max over all objects at once
    std = lc['psfMag'].groupby(level=0).std()
    
    # objects with no observations in this band get nan automatically
    std.name = 'std'
    return df.join(std)

def dust_correction_single_band(df, band):
    A_band = df['ebv'] * tem['dust'][band]
    corrected = (df[f'{band}_psfMag'] - A_band)
    df[f'{band}_psfMagExt'] = corrected
    return df

def dust_correction(df):    
    for band in tem['dust'].keys():
        df = dust_correction_single_band(df, band)
    return df

# Compiles various filtering functions into one
def filtering(df):
    # dust correction
    df = dust_correction(df)

    # color cuts
    ug_query = 'u_psfMagExt - g_psfMagExt > 0.500 and u_psfMagExt - g_psfMagExt < 1.4'
    gr_query = 'g_psfMagExt - r_psfMagExt > -0.15 and g_psfMagExt - r_psfMagExt < 0.4'
    ri_query = 'r_psfMagExt - i_psfMagExt > -0.25 and r_psfMagExt - i_psfMagExt < 0.3'
    iz_query = 'i_psfMagExt - z_psfMagExt > -0.21 and i_psfMagExt - z_psfMagExt < 0.45'
    zy_query = 'z_psfMagExt - y_psfMagExt > -0.22 and z_psfMagExt - y_psfMagExt < 0.15'
    color_query = f'{ug_query} and {gr_query} and {ri_query} and {iz_query} and {zy_query}'
    df = df.query(color_query)

    # quality flag cuts
    flag_query = " and ".join(f"{col} == False" for col in FLAG_COLS)
    df = df.query(flag_query)

    # lc length cut
    df = count_points(df).query("n_lc >= 20")

    # variability cut
    df = calc_std_partition(df).query('std >= 0.1')
    return df

In [6]:
filtered = dp2.map_partitions(filtering)
#filtered = filtered.prune_empty_partitions()

# prune columns for downstream memory management
#filtered = filtered.drop(FLAG_COLS)
filtered

,objectId,coord_dec,coord_ra,objectForcedSource,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv,u_psfMagExt,g_psfMagExt,r_psfMagExt,i_psfMagExt,z_psfMagExt,y_psfMagExt,n_lc,std
npartitions=29184,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 6, Pixel: 28753",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],float64
"Order: 6, Pixel: 28755",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 196331",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 196332",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [7]:
%%time
# check number of surviving objects
def check_len(df):
    return len(df)
lens = filtered.map_partitions(check_len).compute()
lens["result"].sum()

/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 17.40 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


CPU times: user 14min 59s, sys: 27 s, total: 15min 26s
Wall time: 37min 36s


np.int64(993654)

## 2. Run Fitting

In [8]:
def template_fitting(tem, lc, print_outputs = False, fit_n = 20, coeff_n = 10, omega_n = 20, period_range=[0.2, 0.9], cols=['midpointMjdTai', 'band', 'psfMag', 'psfMagErr']):
    '''
    Returns a dictionary with coeffs (mu, d, a, phi), pests (top 3), cov (of linear params: mu (distance modulus), d (dust), a (amplitude)), sigma_P (local uncertainty of a given period), 
        like_P (global posterior likelihood uncertainty), the next best 3 periods, gen_lc (the generated light curve)
    '''
    # compute best coefficients
    omegas = np.arange(1/period_range[-1], 1/period_range[0], 0.1/omega_n) #periods from [0.2, 0.9]: frequencies from [1.1, 5.0]

    lc_bands = list(lc[cols[1]].unique())
    # print(f"template fitting lc_bands: {lc_bands}")
    
    
    rss = mbft.FitTemplate_multiband(tem, lc, omegas, NN=fit_n, use_errors=True, use_dust=False, use_band_shift=True, cols = cols)
    rss = np.array(rss)
    
    best_omega = omegas[np.argmin(rss)]
    best_pest = 1/best_omega
    
    coeffs, cov = mbft.ComputeCoeffsAndCov_multiband(tem, lc, float(best_omega), NN=coeff_n, use_errors=True, use_dust=False, cols = cols)

    # calculate error and posterior on period
    chi2_min = np.min(rss) # rss is chi2 bc it's already scaled by weights

    chi2_red = chi2_min / (len(lc.midpointMjdTai) - len(coeffs)) # length - dof
        
    periods = 1/np.array(omegas)
    rms = np.sqrt(rss / (len(lc.midpointMjdTai) - len(coeffs))) # length - dof
    mask = rss <= chi2_min + 2.3
   
    sigma_P = 0.5 * np.abs(periods[mask][0] - periods[mask][-1])

    # now get posterior likelihood to figure out how likely this is to be the right answer
    peaks, props = find_peaks(-rss, prominence=10, width=0.1)
    try:
        next_best_idx = np.argpartition(props['prominences'], -3)[-3:]
        next_best = periods[peaks[next_best_idx]]
        next_best_chi = rss[peaks[next_best_idx]]
    except:
        next_best = []

    L = np.exp(-0.5 * (rss - rss.min()))  # subtract min to prevent underflow
    norm = np.trapezoid(L, periods) 
    if norm == 0 or ~np.isfinite(norm):
        like_P = np.nan
    else:
        L /= norm # prob density
        P_mean = np.trapezoid(periods * L, periods) # mean of posterior
        P_var  = np.trapezoid((periods - P_mean)**2 * L, periods) # variance on posterior
        like_P = np.sqrt(P_var) # std/likelihood of posterior

    if print_outputs:
        print("omega_best:", best_omega)
        print(f"pest: {best_pest:0.4f} +- {sigma_P:0.6f}, global uncertainty: {like_P:0.4f}")
        print("coeffs (mu, d, a, phi):", coeffs)  # [mu, d, a, phi]
        # print("cov (mu, d, a):\n", cov)

    return dict({'coeffs':coeffs, 'p_est':best_pest, 'cov':cov, 'variance':sigma_P, 'posterior':like_P, 'next_best':next_best, 'rss':rss})

# fit_n is the number of steps used for fitting the period; coeff_n is the number of steps used when finding coefficients
# omega_n is the number of 0.1 period bins for finding the best period
BANDS = ['u', 'g', 'r', 'i', 'z', 'y']  # fixed order, defined at module level

def fit_stat_df(cdf, nested_column='objectForcedSource', fit_n=20, coeff_n=10, omega_n=20, period_range=(0.2, 0.9)):

    # scalar output columns
    for col in ["wrms", "p_est", "p_err", "wrms_ratio", "mu", "d", "a", "phi"]:
        cdf[col] = np.nan

    for band in BANDS:
        for prefix in ["chi2", "flat_chi2", "wrms", "offset"]:
            cdf[f"{prefix}_{band}"] = np.nan
    cdf["chi2_total"] = np.nan
    cdf["flat_chi2_total"] = np.nan

    # per-band columns — one per band per stat
    # discover bands from the first valid lc
    all_bands = []
    for _, row in cdf.iterrows():
        lc_raw = row[nested_column].dropna(subset=['psfMag', 'psfMagErr', 'midpointMjdTai'])
        bands = sorted(lc_raw['band'].unique())
        if bands:
            all_bands = bands
            break

    for band in all_bands:
        for prefix in ["chi2", "flat_chi2", "wrms", "offset"]:
            cdf[f"{prefix}_{band}"] = np.nan
    cdf["chi2_total"] = np.nan
    cdf["flat_chi2_total"] = np.nan

    for idx, row in cdf.iterrows():
        lc_raw = row[nested_column].dropna(subset=['psfMag', 'psfMagErr', 'midpointMjdTai'])
        flag_cols = [c for c in lc_raw.columns if 'flag' in c.lower()]
        lc_flag = lc_raw[~lc_raw[flag_cols].any(axis=1)]

        band_counts = lc_flag.groupby('band')['psfMag'].count()
        valid_bands = band_counts[band_counts >= 3].index
        lc = lc_flag[lc_flag['band'].isin(valid_bands)].reset_index(drop=True)
        lc_bands = list(lc['band'].sort_values(kind='stable').unique())

        if len(lc_bands) <= 1 or len(lc) < 10:
            continue

        pipeline_output = template_fitting(
            tem, lc, fit_n=fit_n, coeff_n=coeff_n, omega_n=omega_n,
            period_range=list(period_range), print_outputs=False,
        )

        coeffs = pipeline_output['coeffs']
        pest   = pipeline_output['p_est']

        cdf.loc[idx, 'p_est'] = pest
        cdf.loc[idx, 'p_err'] = pipeline_output['posterior']
        cdf.loc[idx, 'mu']    = coeffs[0]
        cdf.loc[idx, 'd']     = coeffs[1]
        cdf.loc[idx, 'a']     = coeffs[2]
        cdf.loc[idx, 'phi']   = coeffs[3]

        tem_plot    = mbft.reorder_template_for_lc(tem, lc_bands)
        gamma       = tem_plot['templates']
        t           = tem_plot['temp_time']
        abs_mag_est = tem_plot['abs_mag'](pest, tem_plot)[0]
        model_err   = tem['model_error']['g']

        chi2_total      = 0.0
        flat_chi2_total = 0.0
        wrms_list       = []
        scatter_ratios  = []
        band_sizes      = []

        for i, band in enumerate(lc_bands):
            offset  = coeffs[4 + i] if i < len(lc_bands) - 1 else 0.0
            m_est   = coeffs[0] + abs_mag_est[i] + coeffs[2] * gamma[i, :] + offset
            band_lc = lc[lc['band'] == band]
            minterp = np.interp(band_lc.midpointMjdTai % pest / pest, t, m_est)

            err_total = np.sqrt(band_lc.psfMagErr ** 2 + model_err ** 2)
            weights   = 1.0 / err_total ** 2

            resid  = band_lc.psfMag - minterp
            chi2   = float(np.sum(resid ** 2 * weights))
            chi2_total += chi2

            flat       = np.average(band_lc.psfMag, weights=weights)
            flat_resid = band_lc.psfMag - flat
            flat_chi2  = float(np.sum(weights * flat_resid ** 2))
            flat_chi2_total += flat_chi2

            wrms_t = float(np.sqrt(np.average(resid ** 2,      weights=weights)))
            wrms_f = float(np.sqrt(np.average(flat_resid ** 2, weights=weights)))
            wrms_list.append(wrms_t)
            scatter_ratios.append(wrms_t / wrms_f)
            band_sizes.append(len(band_lc))

            cdf.loc[idx, f"chi2_{band}"]      = chi2
            cdf.loc[idx, f"flat_chi2_{band}"] = flat_chi2
            cdf.loc[idx, f"wrms_{band}"]      = wrms_t
            cdf.loc[idx, f"offset_{band}"]    = offset

        dof      = len(lc) - len(coeffs)
        flat_dof = len(lc) - len(lc_bands)
        cdf.loc[idx, 'chi2_total']      = chi2_total / dof
        cdf.loc[idx, 'flat_chi2_total'] = flat_chi2_total / flat_dof
        cdf.loc[idx, 'wrms']            = float(np.average(wrms_list,     weights=band_sizes))
        cdf.loc[idx, 'wrms_ratio']      = float(np.average(scatter_ratios, weights=band_sizes))

    cdf = cdf.drop(columns="objectForcedSource")

    return cdf

In [25]:
# Meta finding: define the output dataframe by running on a small subset
meta_finder = filtered.head(5)
meta = fit_stat_df(meta_finder)

meta.to_parquet("meta.parquet")
meta

,objectId,coord_dec,coord_ra,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv,u_psfMagExt,g_psfMagExt,r_psfMagExt,i_psfMagExt,z_psfMagExt,y_psfMagExt,n_lc,std,wrms,p_est,p_err,wrms_ratio,mu,d,a,phi,chi2_u,flat_chi2_u,wrms_u,offset_u,chi2_g,flat_chi2_g,wrms_g,offset_g,chi2_r,flat_chi2_r,wrms_r,offset_r,chi2_i,flat_chi2_i,wrms_i,offset_i,chi2_z,flat_chi2_z,wrms_z,offset_z,chi2_y,flat_chi2_y,wrms_y,offset_y,chi2_total,flat_chi2_total
_healpix_29,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3376984963737613727,756398235370674934,-26.890426,300.468557,24.686102,0.490821,23.179617,0.091978,22.777035,0.046577,22.454103,0.040168,22.189194,0.055476,22.028069,0.161418,0.101144,24.204936,22.809371,22.503806,22.246387,22.028364,21.895801,44,0.119607,0.117370,0.242653,0.203308,0.805481,21.260865,0.0,0.431844,0.991260,NaN,NaN,NaN,NaN,0.540004,2.747411,0.043333,1.416932,0.468501,3.241046,0.032836,0.841885,5.813014,6.816769,0.112881,0.399378,5.472863,6.088959,0.147241,0.0,2.607928,2.343626,0.217065,-0.754491,0.480720,0.624642
3376985823765776980,756398304090130592,-26.919674,300.284948,25.44828,1.692292,24.019032,0.109384,23.976486,0.106029,23.67798,0.117796,23.290915,0.15502,23.156652,0.455178,0.107151,24.938541,23.626799,23.687033,23.457929,23.120534,23.016531,46,0.379082,0.334122,0.433630,0.193311,1.379023,22.373316,0.0,1.136336,0.760658,NaN,NaN,NaN,NaN,11.437790,4.359553,0.381813,1.039493,8.143642,6.860862,0.329602,0.888948,5.474948,1.529102,0.274727,0.370913,4.631769,3.734141,0.320822,0.0,2.610709,1.662534,0.414393,-0.913964,1.196254,0.604873
3376985929468093244,756398304090151096,-26.908395,300.321402,21.546488,0.033452,20.622412,0.006356,20.195126,0.004986,20.050941,0.005021,19.972898,0.009373,19.942406,0.026228,0.104723,21.048299,20.239067,19.912231,19.835876,19.806379,19.805459,43,0.127246,0.183647,0.322986,0.013351,1.539267,18.974797,0.0,0.421503,0.629099,NaN,NaN,NaN,NaN,83.383194,20.162750,0.230091,0.782804,69.096779,10.130950,0.201579,0.336780,160.437736,117.162048,0.210059,0.060978,20.175279,8.086065,0.089219,0.0,70.011060,69.795769,0.205100,-0.303179,12.215274,6.259377
3376986016004562941,756398304090151247,-26.906101,300.298303,24.159338,0.315779,22.888605,0.039734,22.438261,0.026403,22.334686,0.034683,22.187532,0.053554,22.346146,0.35007,0.10586,23.65574,22.501098,22.152295,22.117286,22.019205,22.207712,42,0.178257,0.144980,0.222167,0.209215,1.107367,21.125544,0.0,0.389765,0.892171,NaN,NaN,NaN,NaN,7.765332,3.488148,0.129395,1.249969,8.132525,3.462967,0.109609,0.621183,10.676264,14.726497,0.127802,0.217583,9.914908,10.972150,0.170103,0.0,1.751796,1.497829,0.208805,-0.812653,1.233575,1.004341
3376986084556872796,756398304090151886,-26.886131,300.318667,21.804214,0.038755,20.917109,0.007472,20.655981,0.00632,20.653076,0.007966,20.32091,0.009332,20.170815,0.029947,0.105286,21.303348,20.531704,20.371566,20.436855,20.153495,20.033132,44,0.204351,0.360046,0.346487,0.000040,1.496243,19.414476,0.0,0.946980,0.463816,NaN,NaN,NaN,NaN,182.394962,50.890762,0.346934,0.638212,187.215089,51.827129,0.313957,0.357979,690.672148,354.494690,0.450038,0.127189,180.203259,109.310501,0.256771,0.0,219.433877,119.589165,0.409413,-0.346947,40.553315,17.592622


In [9]:
meta=npd.read_parquet("meta.parquet").drop(columns="_healpix_29")
fitted = filtered.map_partitions(fit_stat_df, meta=npd.NestedFrame(
        {col: pd.Series(dtype=dt) for col, dt in meta.dtypes.items()}
    ))
fitted

,objectId,coord_dec,coord_ra,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv,u_psfMagExt,g_psfMagExt,r_psfMagExt,i_psfMagExt,z_psfMagExt,y_psfMagExt,n_lc,std,wrms,p_est,p_err,wrms_ratio,mu,d,a,phi,chi2_u,flat_chi2_u,wrms_u,offset_u,chi2_g,flat_chi2_g,wrms_g,offset_g,chi2_r,flat_chi2_r,wrms_r,offset_r,chi2_i,flat_chi2_i,wrms_i,offset_i,chi2_z,flat_chi2_z,wrms_z,offset_z,chi2_y,flat_chi2_y,wrms_y,offset_y,chi2_total,flat_chi2_total
npartitions=29184,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 6, Pixel: 28753",int64[pyarrow],double[pyarrow],double[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],float[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow]
"Order: 6, Pixel: 28755",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 196331",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 196332",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


# 3. Write out Results

In [10]:
# Save to catalog
# cone search radius = 7000
#fitted.write_catalog("../dp2_fitted_small", overwrite=True)

# cone search radius = 70000
fitted.write_catalog("../dp2_fitted_medium", overwrite=True)

/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 23.97 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/astro/users/brantd/.conda/envs/lsdb_operations/lib/python3.13/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 24.08 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [27]:
# To verify locally -- watch out for results that are too large
result = fitted.compute()
result

/astro/store/epyc/users/brantd/lsdb/src/lsdb/catalog/dataset/healpix_dataset.py:605: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(result)


,objectId,coord_dec,coord_ra,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv,u_psfMagExt,g_psfMagExt,r_psfMagExt,i_psfMagExt,z_psfMagExt,y_psfMagExt,n_lc,std,wrms,p_est,p_err,wrms_ratio,mu,d,a,phi,chi2_u,flat_chi2_u,wrms_u,offset_u,chi2_g,flat_chi2_g,wrms_g,offset_g,chi2_r,flat_chi2_r,wrms_r,offset_r,chi2_i,flat_chi2_i,wrms_i,offset_i,chi2_z,flat_chi2_z,wrms_z,offset_z,chi2_y,flat_chi2_y,wrms_y,offset_y,chi2_total,flat_chi2_total
_healpix_29,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3376984963737613727,756398235370674934,-26.890426,300.468557,24.686102,0.490821,23.179617,0.091978,22.777035,0.046577,22.454103,0.040168,22.189194,0.055476,22.028069,0.161418,0.101144,24.204936,22.809371,22.503806,22.246387,22.028364,21.895801,44,0.119607,0.117370,0.242653,0.205096,0.805481,21.260865,0.0,0.431844,0.991260,NaN,NaN,NaN,NaN,0.540004,2.747411,0.043333,1.416932,0.468501,3.241046,0.032836,0.841885,5.813014,6.816769,0.112881,0.399378,5.472864,6.088959,0.147241,0.0,2.607927,2.343626,0.217065,-0.754491,0.480720,0.624642
3376985823765776980,756398304090130592,-26.919674,300.284948,25.44828,1.692292,24.019032,0.109384,23.976486,0.106029,23.67798,0.117796,23.290915,0.15502,23.156652,0.455178,0.107151,24.938541,23.626799,23.687033,23.457929,23.120534,23.016531,46,0.379082,0.334102,0.433630,0.192412,1.378988,22.373111,0.0,1.136961,0.760754,NaN,NaN,NaN,NaN,11.414526,4.359553,0.381425,1.039146,8.142320,6.860862,0.329575,0.888994,5.478978,1.529102,0.274828,0.370940,4.632827,3.734141,0.320858,0.0,2.611367,1.662534,0.414445,-0.913993,1.195556,0.604873
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3425948724229215012,760957703932627533,-23.33831,301.029913,23.825787,0.208047,22.340546,0.024979,21.935114,0.017043,21.696875,0.037286,21.469999,0.025587,21.342047,0.067632,0.131035,23.202425,21.860883,21.58114,21.427773,21.261641,21.170692,54,0.112501,0.133156,0.484001,0.052731,0.841718,20.768870,0.0,0.605668,0.987811,NaN,NaN,NaN,NaN,3.000203,12.563577,0.056174,0.859457,17.754686,11.240580,0.113624,0.480982,41.949557,74.785439,0.168247,0.131138,18.632762,34.068558,0.125294,0.0,8.304341,8.849518,0.169223,-0.189085,2.037308,3.010801
3425949069530336490,760957703932627460,-23.333927,300.986155,23.600267,0.212599,22.312449,0.019727,22.021017,0.0159,21.797337,0.053147,21.521364,0.032081,21.392597,0.055506,0.130234,22.980717,21.835719,21.669208,21.529881,21.31428,21.22229,41,0.727661,0.593800,0.469239,0.007038,0.861522,19.605743,0.0,2.482951,0.956588,NaN,NaN,NaN,NaN,275.219257,690.124939,0.407358,1.013344,810.190830,672.484192,0.612506,0.743146,NaN,NaN,NaN,NaN,794.084751,1350.884888,0.608659,0.0,678.701831,921.942993,0.680231,-0.473267,79.943646,103.869629
